In [ ]:
from jax import random
import pandas as pd
from datetime import datetime, timedelta
from numpyro import distributions as dist
from numpyro import infer
import yaml as yml
import pycountry

from emu_renewal.inputs import BASE_PATH, DATA_PATH, get_who_targets, get_hosp_target, get_standard_priors
from emu_renewal.inputs import get_european_var_props, get_filtered_seroprev, get_worldbank_national_pop, get_country_mobility, get_standard_targets, get_euro_var_inputs
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import store_outputs
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
# Standard inputs
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
proc_update_freq = 7
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2021, 4, 1)
seed_duration = 10
analysis_to_data_delay = 14
iterations = 1000

# Loaded inputs
country_inputs = yml.safe_load(open(BASE_PATH / "data/config/inputs.yml", "r"))
priors = get_standard_priors()

In [ ]:
for country in ["Italy"]:
    print(country)
    analysis_time = datetime.now().strftime("%Y%m%d_%H%M")
    iso2 = pycountry.countries.lookup(country).alpha_2
    iso3 = pycountry.countries.lookup(country).alpha_3
    mobility = get_country_mobility(iso3)
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(country, analysis_start, analysis_end, init_duration, analysis_to_data_delay)
    strains = ["eu", "alpha"]
    var_target, seed_times = get_euro_var_inputs(country, strains, analysis_start, country_inputs, seed_duration, var_target_start_date, var_target_end_date)
    
    # Collate targets
    seroprev_target_dict = {"seropos": StandardDispTarget(seroprev_target, weight=20.0)} if any(seroprev_target) else {}
    all_targets = {
        "weekly_cases": StandardDispTarget(cases_target, weight=20.0),
        "weekly_deaths": StandardDispTarget(deaths_target, weight=20.0),
        "prop_eu": StandardDispTarget(var_target, weight=20.0),
        "occupancy": StandardDispTarget(hosp_target, weight=20.0),
    } | seroprev_target_dict
    
    for mob_analysis_type in mobility:
        print(mob_analysis_type)
        
        # Define model and fitter
        model = MultiStrainModel(
            get_worldbank_national_pop(iso3),
            analysis_start,
            analysis_end,
            proc_update_freq,
            CosineMultiCurve(),
            GammaDens(),
            init_duration,
            init_data,
            GammaDens(),
            GammaDens(),
            strains,
            "eu",
            seed_times,
            100.0,
            mobility[mob_analysis_type].dropna(),
        )
        
        # Run calibration
        calib = StandardCalib(model, priors, all_targets, proc_dispersion=dist.HalfNormal(0.5))
        kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.1))
        mcmc = infer.MCMC(kernel, num_chains=4, num_samples=iterations, num_warmup=iterations)
        mcmc.run(random.PRNGKey(0), extra_fields=["potential_energy"])
        store_outputs(country, mob_analysis_type, analysis_time, model, calib, mcmc)